In [6]:
from pyspark.sql.functions import col, count, avg, sum as _sum, when

StatementMeta(, cda322f7-255d-4065-bb3d-73bb340e0e77, 8, Finished, Available, Finished, False)

#### Gold table 1: course performance

In [7]:
course_perf_df = spark.sql("""
SELECT
    c.course_id,
    c.course_name,
    c.category,
    COUNT(a.assessment_id) AS total_enrollments,
    AVG(CASE WHEN a.completion_status = 'Completed' THEN 1.0 ELSE 0.0 END) AS completion_rate,
    AVG(a.score) AS avg_score,
    AVG(a.time_spent_hours) AS avg_time_spent_hours,
    AVG(a.feedback_rating) AS avg_feedback_rating,
    SUM(CASE WHEN a.certificate_issued = TRUE THEN 1 ELSE 0 END) AS certificate_issued_count,
    AVG(CASE WHEN a.completion_status = 'Dropped' THEN 1.0 ELSE 0.0 END) AS drop_rate
FROM silver_courses c
JOIN silver_assessments a
    ON c.course_id = a.course_id
GROUP BY c.course_id, c.course_name, c.category
""")

StatementMeta(, cda322f7-255d-4065-bb3d-73bb340e0e77, 9, Finished, Available, Finished, False)

In [8]:
course_perf_df.write.mode("overwrite").saveAsTable("gold_course_performance")
print(f"gold_course_performance rows: {course_perf_df.count()}")

StatementMeta(, cda322f7-255d-4065-bb3d-73bb340e0e77, 10, Finished, Available, Finished, False)

gold_course_performance rows: 100


#### Gold table 2: student engagement

In [9]:
student_eng_df = spark.sql("""
SELECT 
    s.student_id,
    s.subscription_type,
    s.country,
    s.registration_date,
    COUNT(a.assessment_id) AS courses_enrolled,
    SUM(CASE WHEN a.completion_status='Completed' THEN 1 ELSE 0 END) AS courses_completed,
    AVG(a.score) AS avg_score,
    SUM(a.time_spent_hours) AS total_time_spent_hours,
    SUM(CASE WHEN a.certificate_issued THEN 1 ELSE 0 END) AS certificates_earned
FROM silver_students s
JOIN silver_assessments a
ON s.student_id = a.student_id
GROUP BY 
    s.student_id,
    s.subscription_type,
    s.country,
    s.registration_date
""")

StatementMeta(, cda322f7-255d-4065-bb3d-73bb340e0e77, 11, Finished, Available, Finished, False)

In [12]:
student_eng_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_student_engagement")
print(f"gold_student_engagement rows: {student_eng_df.count()}")

StatementMeta(, cda322f7-255d-4065-bb3d-73bb340e0e77, 14, Finished, Available, Finished, False)

gold_student_engagement rows: 500
